# 诗歌生成

# 数据处理

In [6]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow.python import keras as ks
from tensorflow.python.keras import layers
from tensorflow.python.keras import optimizers

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r',encoding='utf-8', errors='replace') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [7]:
class myRNNModel(ks.Model):
    def __init__(self, w2id, cell_type='rnn', num_layers=1):
        """支持多类型RNN单元和多层堆叠的诗歌生成模型
        Args:
            w2id: 词表字典（词到ID的映射）
            cell_type: RNN类型，可选 'rnn', 'gru', 'lstm'
            num_layers: RNN层数（默认为1）
        """
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.num_layers = num_layers
        self.cell_type = cell_type.lower()
        
         # 1. 词嵌入层
        self.embed_layer = ks.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        # 2. 构建多层RNN单元（优化点：支持LSTM/GRU和多层）
        self.rnn_cells = []
        for _ in range(num_layers):
            if self.cell_type == 'lstm':
                cell = ks.layers.LSTMCell(128)
            elif self.cell_type == 'gru':
                cell = ks.layers.GRUCell(128)
            else:  # 默认使用简单RNN
                cell = ks.layers.SimpleRNNCell(128)
            self.rnn_cells.append(cell)
        
        # 3. RNN层（保留原始变量名rnn_layer，增加动态处理）
        self.rnn_layer = ks.layers.RNN(
            self.rnn_cells, return_sequences=True, dynamic=True
        )
        
         # 4. 输出层
        self.dense = ks.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        embed = self.embed_layer(inp_ids)  # 获取词嵌入
        rnn_out = self.rnn_layer(embed)  # RNN前向传播
        logits = self.dense(rnn_out)  # 输出层
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        inp_emb = self.embed_layer(x)  # [batch_size] -> [batch_size, 64]
        new_states = []  # 存储每一层的新状态
        h = inp_emb
        
        # 逐层计算RNN输出（优化点：处理多层状态）
        for i, cell in enumerate(self.rnn_cells):
            h, layer_state = cell(h, state[i])  # 第i层的输入和状态
            new_states.append(layer_state)
        
        logits = self.dense(h)  # [batch_size, vocab_size]
        out = tf.argmax(logits, axis=-1)  # [batch_size]
        return out, new_states
    
    
    def get_initial_state(self, batch_size):
        """生成多层RNN的初始状态（新增方法，用于生成模式）"""
        initial_states = []
        for cell in self.rnn_cells:
            # 每层的初始状态（LSTM为(h, c)，其他RNN类型为h）
            initial_state = cell.get_initial_state(
                batch_size=batch_size, dtype=tf.float32
            )
            initial_states.append(initial_state)
        return initial_states
    

## 一个计算sequence loss的辅助函数，只需了解用途。

In [8]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [9]:
@tf.function
def compute_loss(logits, labels, seqlen):
    """计算基于序列实际长度的掩码平均损失（优化版）"""
    # 输入形状检查（调试用）
    tf.debugging.assert_equal(tf.shape(logits)[:2], tf.shape(labels), 
        message="Logits 和 Labels 的前两维（batch_size, seq_len）必须相同")
    
    # 计算交叉熵损失（自动处理稀疏标签）
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=labels, logits=logits
    )  # shape = [batch_size, seq_len]
    
    # 使用优化后的 reduce_avg 计算每个样本的平均损失
    avg_loss_per_sample = reduce_avg(losses, seqlen, dim=1)  # shape = [batch_size]
    
    # 返回批次平均损失（对标量）
    return tf.reduce_mean(avg_loss_per_sample)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [10]:
optimizer = optimizers.adam_v2.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.821872
epoch 1 : loss 6.6152973
epoch 2 : loss 6.224346
epoch 3 : loss 5.8422074
epoch 4 : loss 5.8816338
epoch 5 : loss 5.504652
epoch 6 : loss 5.53969
epoch 7 : loss 5.418832
epoch 8 : loss 5.341865
epoch 9 : loss 5.305239


# 生成过程

In [18]:
import random
import tensorflow as tf

def generate_poem(begin_word, model, w2id, id2w, poem_type=None):
    """生成四句诗（五言或七言），支持过滤无效符号和强制结构控制"""
    # 随机选择诗类型（若未指定）
    if poem_type is None:
        poem_type = random.choice([5, 7])
    
    # 初始化状态（适配单层或多层RNN）
    if hasattr(model, 'get_initial_state'):
        state = model.get_initial_state(batch_size=1)  # 多层RNN状态
    else:
        state = model.rnncell.get_initial_state(batch_size=1, dtype=tf.float32)  # 单层RNN状态
    
    current_word = tf.constant([w2id[begin_word]], dtype=tf.int32)
    poem_lines = []
    current_line = [begin_word]
    
    # 过滤列表（不加入诗句的符号）
    banned_tokens = ['，', '。','_', 'eos', 'UNK', 'PAD']
    max_attempts = 4 * poem_type * 4  # 最大生成次数
    
    for _ in range(max_attempts):
        current_word, state = model.get_next_token(current_word, state)
        word = id2w.get(current_word.numpy()[0], 'UNK')
        
        # 跳过无效符号
        if word in banned_tokens:
            continue
        
        current_line.append(word)
        
        # 当当前句达到规定字数时处理
        if len(current_line) == poem_type:
            punctuation = '，' if len(poem_lines) < 3 else '。'
            poem_lines.append(''.join(current_line) + punctuation)
            current_line = []  # 重置当前句
            
            # 生成四句后终止
            if len(poem_lines) >= 4:
                break
    
    # 补全未完成的句子
    if len(poem_lines) < 4:
        # 补全前三句用逗号，第四句用句号
        while len(poem_lines) < 4:
            if current_line:
                current_line = current_line[:poem_type]  # 截断至规定长度
                punctuation = '。' if len(poem_lines) == 3 else '，'
                poem_lines.append(''.join(current_line) + punctuation)
                current_line = []
            else:
                # 无剩余词汇时填充默认词（如重复开头词）
                poem_lines.append(begin_word * poem_type + ('。' if len(poem_lines)==3 else '，'))
    
    # 格式修正（确保标点正确）
    poem = ''.join(poem_lines[:4])
    poem = poem.replace('，，', '，').replace('。。', '。')  # 清理多余标点
    return poem

# 示例调用
begin_words = ["日", "红", "山", "夜", "湖", "海", "月"]
for word in begin_words:
    poem = generate_poem(word, model, word2id, id2word)
    print(f"【{word}】开头生成的诗：\n{poem}\n")

【日】开头生成的诗：
日暮山上一，枝声人不可，见不得不知，君有无人事。

【红】开头生成的诗：
红氲鶒萏滨蓉蓥，蓉畔畔递湘峡畔，鹧蓉濆畔黐蓉畔，柚矼跎畔榇啼湘。

【山】开头生成的诗：
山上水云深山风，雨落山水水中春，马无人去春风入，水中来无限路何。

【夜】开头生成的诗：
夜暮不得一年春，马无人去春风入，水中来无处处不，得不知君有无人。

【湖】开头生成的诗：
湖上水云声有春，风起春风落日深，风开树色风雨落，花声马无人去春。

【海】开头生成的诗：
海上山中不，得无人有不，知不得无人，皎然不得无。

【月】开头生成的诗：
月风落花中，生不得不得，无人皎然不，得无人皎然。

